In [2]:
import sys; sys.path.insert(0, '..')
from src.ingest.rama import load_wide, to_long, chronological_split
from src.attack.blind import generar_dataset

long = to_long(load_wide('../data/raw/2025O3.xls'), 'O3')
cca = long[long.station == 'CCA'].reset_index(drop=True)
train, test = chronological_split(cca)

ds = generar_dataset(train, bit=5, tasa=0.05)

print(f"Filas: {len(ds)}   atacadas: {ds.atacado.sum()}")
print(f"\nEfecto del ataque (solo filas atacadas):")
print(ds[ds.atacado == 1].efecto.value_counts().to_string())
print(f"\nDaño: {ds[ds.atacado==1]['daño'].sum():.0f} de {ds.atacado.sum()}")

Filas: 7008   atacadas: 334

Efecto del ataque (solo filas atacadas):
efecto
sin_efecto      237
inflado          66
ocultamiento     31

Daño: 97 de 334


In [3]:
at = ds[ds.atacado == 1].copy()
at['delta'] = at.received_value - at.original_value

print("Direccion del flip (sube o baja el valor):")
print((at.delta > 0).value_counts().to_string())
print(f"\nDelta unico: {at.delta.unique()}")

Direccion del flip (sube o baja el valor):
delta
True     232
False    102

Delta unico: [ 32. -32.]


In [5]:
import pandas as pd
from src.attack.blind import generar_dataset, guardar_csv

resumen = []
for bit in range(13):
    ds = generar_dataset(train, bit=bit, tasa=0.05, seed=42)
    guardar_csv(ds, f'../data/processed/cca_o3_bit{bit:02d}_tasa05.csv')

    at = ds[ds.atacado == 1]
    resumen.append({
        'bit': bit,
        'delta_ppb': 2**bit,
        'atacados': len(at),
        'con_daño': int(at['daño'].sum()),
        'tasa_daño_%': round(at['daño'].mean()*100, 1),
        'inflado': int((at.efecto == 'inflado').sum()),
        'ocultamiento': int((at.efecto == 'ocultamiento').sum()),
    })

pd.DataFrame(resumen)

,bit,delta_ppb,atacados,con_daño,tasa_daño_%,inflado,ocultamiento
0,0,1,334,3,0.9,1,2
1,1,2,334,4,1.2,2,2
2,2,4,334,12,3.6,5,7
3,3,8,334,36,10.8,24,12
4,4,16,334,38,11.4,25,13
5,5,32,334,97,29.0,66,31
6,6,64,334,334,100.0,262,72
7,7,128,334,334,100.0,331,3
8,8,256,334,334,100.0,334,0
9,9,512,334,334,100.0,334,0
